# 第 6 章 · 图谱多跳与路径排序

> 配套 [ch6.html](../ch6.html) · 案例：鲁迅 / 莫言文学知识图谱

## 1. 知识图谱结构

三元组 (头, 关系, 尾) + **关系类型约束**——不是裸 BFS。

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "labs").exists() and (ROOT.parent / "labs").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "labs" / "ch06"))
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS", "DejaVu Sans"]
from IPython.display import display
from reasoning import *
kg = load_kg()
print("查询模式:", kg["query"]["pattern"])
print("标准答案 Y =", kg["query"]["answer_y"])
print("\n边（节选）:")
for h, r, t in kg["edges"][:8]:
    print(f"  ({h}) --[{r}]--> ({t})")

查询模式: ['鲁迅', '创作', '?X', '?X', '发表于', '?Y']
标准答案 Y = 文学周报社

边（节选）:
  (鲁迅) --[创作]--> (狂人日记)
  (鲁迅) --[创作]--> (呐喊)
  (狂人日记) --[发表于]--> (文学周报社)
  (呐喊) --[发表于]--> (文学周报社)
  (狂人日记) --[获得]--> (茅盾文学奖)
  (莫言) --[创作]--> (蛙)
  (莫言) --[创作]--> (红高粱)
  (蛙) --[获得]--> (茅盾文学奖)


## 2. 多跳查询

模板 `(鲁迅,创作,?X)` 且 `(?X,发表于,?Y)` 像 SQL JOIN。

**🤔 预测** · 鲁迅创作的作品有哪些？它们是否都发表于同一期刊？

In [2]:
for line in graph_multihop():
    print(line)

鲁迅创作: 狂人日记, 呐喊
狂人日记 发表于: 文学周报社
呐喊 发表于: 文学周报社
共同发表于 文学周报社: ✓


**⚠️ 误区** · 图谱推理 ≠ 忽略边类型的图搜索——否则会连出「鲁迅→发表于→桌子」这类荒谬路径。

## 3. 软自洽：路径计票排序

缺直接边时，多条间接路径**计票**；硬约束仍守关系类型。

In [3]:
path_ranking()

| 路径 | 得分 |
|------|------|
| 蛙→茅盾文学奖 | 3 |
| 红高粱→典藏 | 2 |
| 红高粱→电影→金熊奖 | 3 |

最高分路径: 蛙→茅盾文学奖


### 思考题
- 若去掉「发表于」约束，会多出哪些错误答案？
- 「莫言→获得→诺奖→代表→红高粱」哪一段是软推断？

实现见 `labs/ch06/reasoning.py`。